# Preprocess KurdiSent Data
Standalone notebook for preprocessing the Kurdish Sentiment dataset using KLPT.

In [4]:
!pip install -q klpt transformers sentencepiece accelerate

In [5]:
import re
import pandas as pd
import os
from google.colab import drive

# Kurdish NLP tools
from klpt.preprocess import Preprocess
from klpt.tokenize import Tokenize
from klpt.stem import Stem

klpt_preprocessor = Preprocess("Sorani", "Arabic")
klpt_tokenizer    = Tokenize("Sorani", "Arabic")
klpt_stemmer      = Stem("Sorani", "Arabic")

In [6]:
drive.mount("/content/drive")

DIR_PATH                    = "/content/drive/MyDrive/google_colab/kusa"
DATA_PATH                   = DIR_PATH + "/datasets"

KURDISENT_PATH              = DATA_PATH + "/KurdiSent.csv"
PREPROCESSED_KURDISENT_PATH = DATA_PATH + "/KurdiSent_preprocessed.csv"

os.makedirs(DATA_PATH, exist_ok=True)

Mounted at /content/drive


In [7]:
# Whitelist
SCHWA_FIXES = {
    "له": "لە",   # le  (preposition)
    "به": "بە",   # be  (preposition)
    "که": "کە",   # ke  (relativiser)
    "نه": "نە",   # ne  (negation)
    "وه": "وە",   # we  (particle)
}

def normalize_schwa(token):
    return SCHWA_FIXES.get(token, token)


MWE_SEP    = "\u2012"   # KLPT-internal separator inside compound tokens
WORD_BOUND = "\u2581"   # KLPT single-word boundary marker


def data_cleaning_ckb(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)

    normalized = klpt_preprocessor.preprocess(text).strip()

    # Surface view: natural text, but the same whole-word schwa fix for alignment.
    surface = " ".join(normalize_schwa(t) for t in normalized.split())

    tokens = klpt_tokenizer.word_tokenize(normalized)

    lemmatized_words = []
    for word in tokens:
        clean_word = word.replace(WORD_BOUND, "")
        for part in clean_word.split(MWE_SEP):      # split compounds first
            part = part.strip()
            if not part:
                continue
            part = normalize_schwa(part)            # whitelist
            try:
                analysis = klpt_stemmer.analyze(part)
                if analysis:
                    lemma_word = analysis[0].get("lemma", part)
                    if isinstance(lemma_word, list):
                        lemma_word = lemma_word[0] if lemma_word else part
                    lemmatized_words.append(str(lemma_word))
                else:
                    lemmatized_words.append(part)    # OOV -> surface fallback
            except Exception:
                lemmatized_words.append(part)

    lemma = " ".join(lemmatized_words)
    return surface, lemma


In [8]:
print("Loading dataset...")
df_ckb = pd.read_csv(KURDISENT_PATH)
if 'num_label' in df_ckb.columns:
    df_ckb['label'] = df_ckb['num_label']

print("Preprocessing...")
df_ckb[["surface", "lemma"]] = df_ckb["text"].apply(
    lambda x: pd.Series(data_cleaning_ckb(str(x)))
)
df_ckb_clean = df_ckb[df_ckb["surface"].str.strip() != ""].reset_index(drop=True)

print("Saving preprocessed data...")
df_ckb_clean.to_csv(PREPROCESSED_KURDISENT_PATH, index=False, encoding="utf-8")
print(f"Saved to: {PREPROCESSED_KURDISENT_PATH}")

Loading dataset...
Preprocessing...
Saving preprocessed data...
Saved to: /content/drive/MyDrive/google_colab/kusa/datasets/KurdiSent_preprocessed.csv


In [9]:
print("Verifying preprocessed data...")
df_ckb_clean = pd.read_csv(PREPROCESSED_KURDISENT_PATH, encoding="utf-8")
df_ckb_clean = df_ckb_clean.loc[:, ~df_ckb_clean.columns.str.contains("^Unnamed")]
df_ckb_clean = df_ckb_clean.dropna(subset=["surface", "lemma"]).reset_index(drop=True)
print(f"Dataset loaded: {len(df_ckb_clean)} instances")
print(df_ckb_clean.head())

Verifying preprocessed data...
Dataset loaded: 12306 instances
   num_label category                                               text  \
0          1      art  هاوڕامە لەگەڵ جەنابت بەڵام بۆ چونم جیاوازە,  ج...   
1          2   social  هیوای ژیانێکی پڕ لە بەختەوەر یم هەیە بۆ تان پی...   
2          2   social       بەدەست خۆم نییە کەبابم پێ لە ساوەر خۆش ترە.    
3          2      art                  دیمەنێکی زۆر جوان و پڕ مانا بوو.    
4          1   social  نەکەی بخوای ناخۆشە  بیگانە دەتوانی دوو قسەی لە...   

   label                                            surface  \
0      1  هاوڕامە لەگەڵ جەنابت بەڵام بۆ چونم جیاوازە, جو...   
1      2  هیوای ژیانێکی پڕ لە بەختەوەر یم هەیە بۆ تان پی...   
2      2        بەدەست خۆم نییە کەبابم پێ لە ساوەر خۆش ترە.   
3      2                   دیمەنێکی زۆر جوان و پڕ مانا بوو.   
4      1  نەکەی بخوای ناخۆشە بیگانە دەتوانی دوو قسەی لەگ...   

                                               lemma  
0  هاوڕامە لەگەڵ جەناب بەڵا بۆ چونم جیاوازە, ج